## Problems with intervals

usually uses the following ways
- 2 values: start, end, to keep track of the overlaps 
- bisect: binary search to insert values in the list / stack
- priority queue
- sweep line -> good if the input is streaming one, so cannot sort all the inputs in advance, for other cases that inputs are already given at once and are sortable, other way is more straightforward

check if two intervals are overlapped

- overlap of 2 interval a b is (max(a0, b0), min(a1, b1))

In [2]:
from typing import List, Optional

<b> 57 insert interval </b>

create a new list as result list. loop over each item in the current list to compare with the new intervals. there are several different use cases
- [o] < [new], no overalp, [o] can be added to the result list, but new interval still need to compare with next item in the list
- [n] < [o], no overlop, [new] can be added to the result list, same as all the rest of the interval items 
- having overlap in [n] and [o], get the updated [new] by combining them, and this new internal is used to compare against next item in the interval list

In [3]:

def insert(intervals: List[List[int]], newInterval: List[int]) -> List[List[int]]:
    res = []
    n = len(intervals)

    for i in range(n):
        old_interval = intervals[i]
        # no overlap
        if old_interval[1] < newInterval[0]:
            res.append(old_interval)
        elif newInterval[1] < old_interval[0]:
            res.append(newInterval)
            res.extend(intervals[i:])
            return res
        else:
            newInterval = min(old_interval[0], newInterval[0]), max(old_interval[1], newInterval[1])
    res.append(newInterval)
    return res

In [4]:
insert([[1,5]], [2,3])

[(1, 5)]

<b> 56 merge intervals </b>

similar idea as above, using 2 vars, one for prev_item, another for next_item, do the comparason and merge as looping over each item. but at the beginning, need to sort the array first

In [5]:
def merge(intervals: List[List[int]]) -> List[List[int]]:
    # sort the intervals by first number
    intervals.sort()
    # edge case when intervals is empty or only have 1 item
    n = len(intervals)
    if n <= 1:
        return intervals
    # compare prev_item with next_item, 
    # if prev_item[1] < next_item[0], no overlap
    # otherwise merge and update the prev_item, moving forward to next next_item
    prev_item = intervals[0]
    res = []
    for i in range(1, n):
        next_item = intervals[i]
        if prev_item[1] < next_item[0]:
            res.append(prev_item)
            prev_item = next_item
        else:
            prev_item = min(prev_item[0], next_item[0]), max(prev_item[1], next_item[1])
    res.append(prev_item)
    return res

but can have a more concise version on space complexity, to reduce the var of prev_item, which is actually the result[-1] item

In [6]:

def merge(intervals: List[List[int]]) -> List[List[int]]:
    """Sorting takes O(n log(n)) and merging the intervals takes O(n). So, the resulting algorithm takes O(n log(n))."""
    """a precise version"""
    intervals.sort()
    result = []

    for item in intervals:
        if not result or result[-1][1]<item[0]:
            # if result is empty (just start)
            # or NO overlapping
            result.append(item)
        else:
            # result[-1][0] no need to be updated because the intervals list is sorted.
            # so prev_item[0] < next_item[0] is always true
            result[-1][1] = max(result[-1][1], item[1])
    return result

In [7]:
merge([[1,3],[2,100],[4, 15]])

[[1, 100]]

<b> 2848 points that intersect with cars </b>

same as merge intervals problem. first merge the intervals then calculate the res.  
the nums array is sorted, so wont have casese like queue [ [1,3], [4,7] ] then add [2,5] which require items in queue to merge again. 

after sorting the nums array, only need to compare with the last item in the queue to consider if need a merging or append a new non-overlapping item


In [26]:

def numberOfPoints(nums: List[List[int]]) -> int:
    """
    merge overlapped values, then calculate the coverred length
    """
    queue = [] # [(val, val)] non-overalpped items
    for start, end in sorted(nums):
        if not queue or queue[-1][1] < start:
            # no overlapping
            queue.append([start, end])
        else:
            # overlapping
            queue[-1][1] = max(queue[-1][1], end)
    res = 0
    for start, end in queue:
        res += (end - start) + 1
    return res




In [27]:
numberOfPoints([[3,6],[1,5],[4,7]])

7

<b> 3169 Count Days Without Meetings </b>

similar to merge interval problems, can merge intervals, then get the total meeting days. but instead of saving a queue of list, we can use a single list to save space

In [29]:

def countDays(days: int, meetings: List[List[int]]) -> int:
    """
    merge intervals among meetings, then calculate the total day of meeting
    res = days - total day of meeting
    """
    prev_meeting = []
    total_meeting_days = 0
    for start, end in sorted(meetings):
        if not prev_meeting:
            prev_meeting = [start, end]
        elif prev_meeting[-1] < start:
          # no overlapping
            total_meeting_days += prev_meeting[1] - prev_meeting[0] + 1
            prev_meeting = [start, end]
        else:
            # overlapping
            prev_meeting[1] = max(prev_meeting[1], end)
    if prev_meeting:
        total_meeting_days += prev_meeting[1] - prev_meeting[0] + 1
    return days - total_meeting_days





In [30]:
countDays(10, [[5,7],[1,3],[9,10]])

2

<b> 495 Teemo Attacking </b>

similar problems too, using prev to check the end time from previous attack, and compare with the next attacking start time to see if there is any overlaop.

Basically same idea of keeping track of start / end, and compare with next item

In [8]:

def findPoisonedDuration(timeSeries: List[int], duration: int) -> int:
    res = 0
    n = len(timeSeries)
    prev = None
    for i in range(n):
        # using None as the prev initial value for starting the first attack
        if prev is not None:
            if prev < timeSeries[i]:
                res += duration
            else:
                res += (timeSeries[i] - timeSeries[i-1])
        # no matter overlapping or not, the new end time is starting from the timeSeries[i]
        prev = timeSeries[i] + duration - 1

    res += duration
    return res

In [9]:
findPoisonedDuration([1,3,4], 2)

5

<b> 715 range module </b>

smart solution to use bisect and bisect_right to determine if the new range has overlap with existing one or not.

also the slice in a python list can be replaced by another list is outside of my expectations.

for example a[2:2] = [3,4] can insert a new list in position 2. and a[1:3] = [] going to remove the item at position 1 and 2.


In [10]:
from bisect import bisect_left, bisect_right
class RangeModule:

    def __init__(self):
        # example [10, 15, 20, 25] means [10, 15), [20, 25)
        # so the number is always even
        self.range = [] 

    def addRange(self, left: int, right: int) -> None:
        i, j = bisect_left(self.range, left), bisect_right(self.range, right)
        sub_stack = []
        if i%2==0:
            # when even, the left value from new range is not covered
            # so need to create a new element for it
            sub_stack.append(left)
        if j%2==0:
            # same as j
            sub_stack.append(right)
        # add the range
        # if i, j all odd, then sub_stack is [], merging existing range
        # if i, j all even, new range is added to a position since now i j should be the same
        # when one of i, j is odd, then it is partial overlap so partial merge
        self.range[i:j] = sub_stack

    def queryRange(self, left: int, right: int) -> bool:
        # here i is using bisect_right, and j is using bisect_left
        # because the right end is not included,
        # for example [10, 15), query for [15, 20) if using bisect_left(l)
        # i is 1 which is odd num meaning partial covered, but 15 is not included
        i, j = bisect_right(self.range, left), bisect_left(self.range, right)
        # when i j is equal, and i%2==0 then completely nothing is found
        # partial coverage is not count and should return False
        # when i j is equal and both %2 =1 means fully covered
        return i == j and i%2 == 1
        

    def removeRange(self, left: int, right: int) -> None:
        i, j = bisect_left(self.range, left), bisect_right(self.range, right)
        sub_stack = []
        if i%2==1:
            # found left is in same existing range
            sub_stack.append(left)
        if j%2==1:
            # found right is in same existing range
            sub_stack.append(right)
        self.range[i:j] = sub_stack # insert into the even position


# Your RangeModule object will be instantiated and called as such:
# obj = RangeModule()
# obj.addRange(left,right)
# param_2 = obj.queryRange(left,right)
# obj.removeRange(left,right)

<b> 252 Meeting Room </b>

simplied version, no need to use heap queue, only need to sort the intervals, and compare if the previous meeting's end time is larger than next meeting's starting time or not 

In [11]:

def canAttendMeetings(intervals):
    """
    :type intervals: List[List[int]]
    :rtype: bool
    """
    # simplified versinon, same as only one 1 available room
    if not intervals or len(intervals)==1:
        return True
    intervals = sorted(intervals)
    prev = intervals[0]
    for i in range(1, len(intervals)):
        # if previous meeting's end time > next meeting's start time
        if prev[1] > intervals[i][0]:
            return False
        prev = intervals[i]
    return True

<b> 253 Meeting Room II </b>

Given an array of meeting time intervals intervals where intervals[i] = [starti, endi], return the minimum number of conference rooms required.


similar as above question, using 2 heap queue to keep track of inused room and available room. we can keep adding number of rooms when there is no available one.

it is still a problem to compare the prev's end time with next start time

In [9]:
from heapq import heappop, heappush

def minMeetingRooms(intervals):
    """
    :type intervals: List[List[int]]
    :rtype: int
    """

    available_room = []
    inused_room = [] # end_time, room_index
    room_num = 0

    # need to sort the intervals first
    for start, end in sorted(intervals):
        # check inused_room to free up rooms
        while inused_room and inused_room[0][0] <= start:
            end_time, room_index = heappop(inused_room)
            heappush(available_room, room_index)

        if available_room:
            # with room available
            room_index = heappop(available_room)
        else:
            # add a new room
            room_index = room_num
            room_num += 1
        heappush(inused_room, (end, room_index))
    return room_num

In [10]:
intervals = [[0,30],[5,10],[15,20]]
minMeetingRooms(intervals)

2

In [15]:
from heapq import heappush, heappop

def minMeetingRooms(intervals: List[List[int]]) -> int:
    queue = [] # only store the end 
    for start, end in sorted(intervals):
        if queue and queue[0] <= start:
            # release an available room
            heappop(queue)
        # add new meetings to the queue
        heappush(queue, end)
    return len(queue)


In [16]:
intervals = [[0,30],[5,10],[15,20]]
minMeetingRooms(intervals)

2

<b> 2406 Divide Intervals Into Minimum Number of Groups </b>

same as meeting room II but with a simpler solution. It returns the minimum number of groups you need to make, same as returning the min number of room needed.

here using a queue and length of queue to represent the number of group


In [11]:
from heapq import heappop, heappush

def minGroups(intervals: List[List[int]]) -> int:
    """
    same to the meeting room II, return the min required meeting room num
    but here simplifed the solution
    """
    queue = [] # saving the right interval only
    for left, right in sorted(intervals):
        if queue and queue[0] < left:
            # can consider it as using the "room" released by 0
            # only need to compare with first item because it is dequeu
            # if queue[0] > current starting number, rest of the numbers in the queue
            # is larger too, 
            heappop(queue)
        # update the queue, no matter using the existing room (already pop out) or using a new room
        heappush(queue, right)
    # len of queue is the number of room used
    return len(queue)

<b> 2402. Meeting Rooms III </b>

leverage 2 priority queues. one to track the available room, queue value is the room index, another to track the in used room, value is the a tuple of (end time, room index).  
loop over each meetings to see which room it should be put. having additional list to keep track of times each room is used.

In [14]:
from heapq import heapify, heappop, heappush
class Solution(object):
    def mostBooked(self, n, meetings):
        """
        :type n: int
        :type meetings: List[List[int]]
        :rtype: int
        """
        # create 2 priority queue
        available_room = [i for i in range(n)] # store room_index
        inused_room = [] # store [end_time, room_index]
        heapify(available_room)
        res = [0] * n
        for start, end in sorted(meetings):
            # checked inused_room to free up room
            # if first inused room finishes the meeting before current meeting start
            # free up the inused room and move it to available queue
            # it is min queue, so first item has the smallest end_time value
            while inused_room and inused_room[0][0] <= start:
                end_time, room_index = heappop(inused_room)
                heappush(available_room, room_index)
            if available_room:
                # if room is available, get the first room with smallest index
                room_index = heappop(available_room)
                heappush(inused_room, [end, room_index])
            else:
                # wait for in used room with nearest ending time to be ready
                end_time, room_index = heappop(inused_room)
                # the first room returned has the earliest end_time
                # the new end_time will be cur val + the meeting duration time
                # this room with new end time will be added back to the queue
                # and the order is updated
                heappush(inused_room, [end_time + end-start, room_index])
            # keep track of the count that each room is used
            res[room_index] += 1
        return res.index(max(res))

In [12]:
minGroups([[5,10],[6,8],[1,5],[2,3],[1,10]])

3

<b> 763 partial label </b>

the problem at the beginning looks confusing, but its rule is to put the same char in the same substring. so the partition is limited to the last index of the char in the string.

It is still the start and end pointers, and keep updating the pointers based on certain criteria

In [20]:

def partitionLabels(S: str) -> List[int]:

    # save the right most index of each char
    hashmap = {c: i for i, c in enumerate(S)}
    result = []
    n = len(S)
    # two pointers here
    start,end = 0, 0

    # construct another pointer
    for i, c in enumerate(S):
        # the possible end position for character c
        # it can happen that the end keep increasing because the char encountered keep further pushing the end
        end = max(end, hashmap[c])
        if i == end:
            # one partition is ready
            result.append(end-start+1)
            # the ith position is included in current partition, so the start of next partition should be i+1
            start = i+1

    return result


<b> 616 add bold tag in string </b>

You are given a string s and an array of strings words.

You should add a closed pair of bold tag \<b> and \</b> to wrap the substrings in s that exist in words.

If two such substrings overlap, you should wrap them together with only one pair of closed bold-tag.
If two substrings wrapped by bold tags are consecutive, you should combine them.
Return s after adding the bold tags.
  
Input: s = "abcxyz123", words = ["abc","123"]  
Output: "\<b>abc\</b>xyz\<b>123\</b>"   

Input: s = "aaabbb", words = ["aa","b"]  
Output: "\<b>aaabbb\</b>" 

using the same pattern of merging intervals. First construct a list of index_pair as intervals. for example:

s = "abcxyz123", words = ["abc","123"], the index_pair should be [(0,3), (6,9)], which indicates that we should insert pair of "\<b>\</b>" in 2 location.
s = "aaabbb", words = ["aa","b"], the index_pair whould be [(0,2), (1, 3), (3,4),(4,5),(5,6)] -> this can actually be merged and become [(0, 6)], which indicates that we should only insert pair of "\<b>\</b>" in 1 location.

there are some tricky part to reconstruct the strings. for example, if the first bold tag pair does not start from the beginning, or end at the end of S. need to include those substring without bold tag pair.

In [ ]:

def addBoldTag(s: str, words: List[str]) -> str:
    # get the index pair of where the <b> should be inserted
    # [(0, 2), (6, 8)] -> no overlap
    # [(0, 1), (1,2), (3, 4), (4, 5)] -> overlap and merge -> [(0,5)]

    index_pair = []
    n = len(s)
    for ind in range(n):
        for w in words:
            m = len(w)
            if s[ind: ind+m] == w:
                if not index_pair or index_pair[-1][-1] < ind:
                    # no overlapping
                    index_pair.append((ind, ind+m))
                else:
                    # merge
                    end_ind = max(index_pair[-1][-1], ind+m)
                    index_pair[-1] = (index_pair[-1][0], end_ind)
    # loop over index_pair to construct the new string
    # print("index_pair", index_pair)
    result = []
    prev_end_ind = 0
    for start, end in index_pair:
        # adding substring that no need <b></b>
        if prev_end_ind < start:
            result.append(s[prev_end_ind: start])               
        result.append("<b>")
        result.append(s[start:end])
        result.append("</b>")
        prev_end_ind = end

    result.append(s[prev_end_ind: n])
    return "".join(result)



<b> 759 employee free time </b>

even though the schedule list is given per employee, it is actually does not matter the interval is from which employee. As long as all intervals are sorted, and the free_time interval can be found

Input: schedule = [[[1,2],[5,6]],[[1,3]],[[4,10]]]  
Output: [[3,4]]  
Explanation: There are a total of three employees, and all common  
free time intervals would be [-inf, 1], [3, 4], [10, inf].  
We discard any intervals that contain inf as they aren't finite.  

In [2]:
"""
# Definition for an Interval.
class Interval:
    def __init__(self, start: int = None, end: int = None):
        self.start = start
        self.end = end
"""


def employeeFreeTime(schedule: '[[Interval]]') -> '[Interval]':
    """
    it does not matter the interval is from which employee, so flatten the schedule
    when compairing prev interval with next interval
    - if overlapped, update the prev interval's range
    - if no overlap, a free_time interval is found and can update the result list, then update the prev
    """
    if not schedule:
        return []
    sorted_intervals = sorted([interval for emp_s in schedule for interval in emp_s], key=lambda x: x.start)
    res = [] # store the free intervals
    prev = sorted_intervals[0]
    for next_interval in sorted_intervals[1:]:
        # if not overlap
        if prev.end < next_interval.start:
            res.append(Interval(prev.end, next_interval.start))
            prev = next_interval
        else: # merge
            prev.end = max(prev.end, next_interval.end)
    return res


<b> 715 range module </b>

smart way of using bisect_left, bisect_right.  
also smart to use a simple list to represent the list of ranges

In [3]:
from bisect import bisect_left, bisect_right
class RangeModule:

    def __init__(self):
        # example [10, 15, 20, 25] means [10, 15), [20, 25)
        # so the number is always even
        self.range = [] 

    def addRange(self, left: int, right: int) -> None:
        i, j = bisect_left(self.range, left), bisect_right(self.range, right)
        sub_stack = []
        if i%2==0:
            # when even, the left value from new range is not covered
            # so need to create a new element for it
            sub_stack.append(left)
        if j%2==0:
            # same as j
            sub_stack.append(right)
        # add the range
        # if i, j all odd, then sub_stack is [], merging existing range
        # if i, j all even, new range is added to a position
        # when one of i, j is odd, then it is partial overlap so partial merge
        self.range[i:j] = sub_stack
        print(self.range)

    def queryRange(self, left: int, right: int) -> bool:
        # here i is using bisect_right, and j is using bisect_left
        # because the right end is not included,
        # for example [10, 15), query for [15, 20) if using bisect_left(l)
        # i is 1 which is odd num meaning partial covered, but 15 is not included
        i, j = bisect_right(self.range, left), bisect_left(self.range, right)
        # when i j is equal, and i%2==0 then completely nothing is found
        # partial coverage is not count and should return False
        # when i j is equal and both %2 =1 means fully covered
        return i == j and i%2 == 1
        

    def removeRange(self, left: int, right: int) -> None:
        i, j = bisect_left(self.range, left), bisect_right(self.range, right)
        sub_stack = []
        if i%2==1:
            # found left is in same existing range
            sub_stack.append(left)
        if j%2==1:
            # found right is in same existing range
            sub_stack.append(right)
        self.range[i:j] = sub_stack # insert into the even position


# Your RangeModule object will be instantiated and called as such:
# obj = RangeModule()
# obj.addRange(left,right)
# param_2 = obj.queryRange(left,right)
# obj.removeRange(left,right)

In [5]:

range_module = RangeModule()
range_module.addRange(10, 15)
range_module.addRange(20, 25)
range_module.addRange(9, 21)

[10, 15]
[10, 15, 20, 25]
[9, 25]


<b> 986 interval list intersection </b>

time complexity is O(N*M) where N, M are the length of the two lists. So only loop over each list once.  
using i, j as index pointer to loop over each item in 2 list, and compare if they have overalop then take actions accordingly

first solution is the initial one, second solution is an optmized one

In [7]:
from typing import List
def intervalIntersection(firstList: List[List[int]], secondList: List[List[int]]) -> List[List[int]]:
    n = len(firstList)
    m = len(secondList)
    if not n or not m:
        return []
    i, j = 0, 0
    res = []
    while i < n and j < m:
        first_item = firstList[i]
        second_item = secondList[j]
        # if not overlap
        if first_item[1] < second_item[0]:
            i+= 1
        elif second_item[1] < first_item[0]:
            j += 1
        # overlap
        else:
            overlap = (max(first_item[0], second_item[0]), min(first_item[1], second_item[1]))
            res.append(overlap)
            if first_item[1] < second_item[1]:
                i += 1
            elif second_item[1] < first_item[1]:
                j += 1
            else:
                i+=1
                j+=1
    return res

In [8]:

def intervalIntersection(firstList: List[List[int]], secondList: List[List[int]]) -> List[List[int]]:
    n = len(firstList)
    m = len(secondList)
    if not n or not m:
        return []
    i, j = 0, 0
    res = []
    while i < n and j < m:
        a_start, a_end = firstList[i]
        b_start, b_end = secondList[j]
        # if overalop
        if (a_end >= b_start) and (b_end >= a_start):
        # same as if not (first_item[1] < second_item[0]) and not (second_item[1] < first_item[0])
        # where not overlap is when first_item[1] < second_item[0] or second_item[1] < first_item[0]
            overlap = (max(a_start, b_start), min(a_end, b_end))
            res.append(overlap)
        # determine which list move forward
        if a_end < b_end:
            i += 1
        else:
            j += 1
    return res

<b> 2446 determine if two events have conflict </b> 

i first transformed the string of time to something that is comparable. But cool to see another way to finding the overlap :  
If these two intervals overlap, it should exist a value x,  
left1 <= x <= right1 && left2 <= x <= right2  
so that  
max(left1, left2) <= x <= min(right1, right 2)  
so that  
left1 <= right2 && left2 <= right1  

or the (not left 1[1]< left 2[0]) and (not left2[1] < left1[0]) is actually transformed into the above expression. 

somehow in python the string comparision of time actually works

In [17]:

def haveConflict(event1: List[str], event2: List[str]) -> bool:

    def helper(event):
        res = [] # [(HH, MM), (HH, MM)]
        for e in event:
            tmp = [int(item) for item in e.split(":")]
            res.append(tmp)
        return res
    event1_time = helper(event1)
    event2_time = helper(event2)
    if (event1_time[1][0] < event2_time[0][0] or event2_time[1][0] < event1_time[0][0]):
        # not overlap
        return False
    elif (event1_time[1][0] == event2_time[0][0] and event1_time[1][1] < event2_time[0][1]) or (event2_time[1][0] == event1_time[0][0] and  event2_time[1][1] < event1_time[0][1]):
        # no overlap
        return False
    return True


In [19]:
event1 = ["01:15","02:00"]
event2= ["02:00","03:00"]
haveConflict(event1, event2)

True

In [20]:
def haveConflict(event1: List[str], event2: List[str]) -> bool:
    # cool to know the time in string is actually comparable in python
    return event1[0] <= event2[1] and event2[0] <= event1[1]

<b> 836 rectangle overlap </b>

similar to the question above

In [22]:

def isRectangleOverlap(rec1: List[int], rec2: List[int]) -> bool:
    """
    if two rectangles overlap, there must exist some point (x, y) that satisfies
    rect1_x1 < x < rect1_x2, rect1_y1 < y < rect1_y2
    rect2_x1 < x < rect2_x2, rect2_y1 < y < rect2_y2
    which is 
    max(rect1_x1, rect2_x1) < x < min(rect1_x2, rect2_x2)
    max(rect1_y1, rect2_y1) < y < min(rect1_y2, rect2_y2)
    same as 
    rect1_x1 < rect2_x2 and rect2_x1 < rect1_x2
    rect1_y1 < rect2_y2 and rect2_y1 < rect1_y2
    """
    rect1_x1, rect1_y1, rect1_x2, rect1_y2 = rec1
    rect2_x1, rect2_y1, rect2_x2, rect2_y2 = rec2
    return (rect1_x1 < rect2_x2) and (rect2_x1 < rect1_x2) and (rect1_y1 < rect2_y2) and (rect2_y1 < rect1_y2)



In [23]:
rec1 = [0,0,2,2]
rec2 = [1,1,3,3]
isRectangleOverlap(rec1, rec2)

True

<b> 2580 Count Ways to Group Overlapping Ranges </b>

finding overlapping as well, but if one item is not overlapped with others, it is free to be put in either group.  
smart to use a single last value to keep track of overlapping or not, together using sorted array

In [24]:

def countWays(ranges: List[List[int]]) -> int:
    res = 1
    last = -1 # store the end value of each group
    mod = 10**9 + 7
    for start, end in sorted(ranges):
        if last < start:
            # no overlap with previous item
            # so the item can choose either group, 2 options
            res = res * 2 % mod
        # if overlapped, then can only be put in one group
        last = max(last, end)
    return res

In [25]:
countWays([[0,0],[8,9],[12,13],[1,3]])

16

### About line sweep algorithms

[blog of explanation](https://leetcode.com/discuss/study-guide/2166045/line-sweep-algorithms)

basic idea of pattern 1:
- add 1 when it starts
- remove 1 then it ends
- save the [start, 1], [end, -1] into a list, then sort it
- loop over the list and add un the second value, compare it with a global min or max value
- imagine a line sweeping from the start to end, at each position check the status and update the status

<b> 1854 Maximum Population Year </b>

exactly follows the above explanation.

In [31]:

def maximumPopulation(logs: List[List[int]]) -> int:
    year = -1
    max_pop = 0
    population = 0
    queue = []
    for birth, death in logs:
        queue.append([birth, 1])
        queue.append([death, -1])
    for item in sorted(queue):
        population += item[1]
        if population > max_pop:
            max_pop = population
            year = item[0]

    return year

In [32]:
maximumPopulation([[1950,1961],[1960,1971],[1970,1981]])

1960

<b> 253 meeting room II </b>

can use the concept of sweep line algo too

In [34]:


def minMeetingRooms(intervals: List[List[int]]) -> int:
    """
    if using sweep line algo,
    get the global max at any given time, that is the min number of room needed
    """
    queue = [] # [(start, 1), (end, -1)]
    for start, end in intervals:
        queue.append([start, 1])
        queue.append([end, -1])
    count = 0 # local count, finally will be 0 since all meeting will end at some time
    res = 0 # store the max count encountered so far
    for item in sorted(queue):
        count += item[1]
        res = max(count, res)
    return res

In [35]:
minMeetingRooms([[0,30],[5,10],[15,20]])

2

<b> 731 my calendar II </b>

using the sweep line algo.  
a dictionary {time: count} to track the start and end of each range.  
then sort the dictionary items, to loop over reach (time, count) to aggregate the count value. if the count > 3 that means there are triple booking and is invalid.  
the -1 of start, +1 of end is to reverse the adding / booking action

here the interval is half open: [start, end)

time complexity:
- the sorting takes O(KlogK) where K is the number of (start, end) pair
- loop over the sorted items takes O(K)
- in total it is O(K) + O(KlogK) but KlogK is dominant, so time complexity is O(KlogK)

In [36]:
from collections import defaultdict
class MyCalendarTwo:

    def __init__(self):
        self.calendar = defaultdict(int)

    def book(self, start: int, end: int) -> bool:
        self.calendar[start] += 1
        self.calendar[end] -= 1
        cur = 0
        for t, val in sorted(self.calendar.items()):
            cur += val
            if cur == 3:
                self.calendar[start] -= 1 # reverse the adding of this meeting
                self.calendar[end] += 1
                return False
        return True


# Your MyCalendarTwo object will be instantiated and called as such:
# obj = MyCalendarTwo()
# param_1 = obj.book(start,end)

In [40]:
# ["MyCalendarTwo", "book", "book", "book", "book", "book", "book"]
# [[], [10, 20], [50, 60], [10, 40], [5, 15], [5, 10], [25, 55]]
calendar = MyCalendarTwo()
print(calendar.book(10, 20))
print(calendar.book(50, 60))
print(calendar.book(10, 40))
print(calendar.book(5, 15))

True
True
True
False


<b> 729 my calendar I </b>

it can use the sweep line algo above, that each add takes O(NLogN) because of sort, but can also simply using the overlapping concept which takes only O(N) for each book function

In [41]:

class MyCalendar:

    def __init__(self):
        self.booking = []

    def book(self, start: int, end: int) -> bool:
        for prev_start, prev_end in self.booking:
            if max(prev_start, start) < min(prev_end, end):
                # with overlap
                return False
        self.booking.append((start, end))
        return True


# Your MyCalendar object will be instantiated and called as such:
# obj = MyCalendar()
# param_1 = obj.book(start,end)

<b> 732 my calendar III </b>

same, can use the sweep line algo. The problems wants to know the max count of booking that overlapped with each other

In [42]:
class MyCalendarThree:

    def __init__(self):
        self.booking = defaultdict(int)
        

    def book(self, startTime: int, endTime: int) -> int:
        # sweep line, and keep track of largest count during loop
        self.booking[startTime] += 1
        self.booking[endTime] -= 1
        count = 0
        max_count = 0
        for t, c in sorted(self.booking.items()):
            count += c
            max_count = max(max_count, count)
        return max_count
        


# Your MyCalendarThree object will be instantiated and called as such:
# obj = MyCalendarThree()
# param_1 = obj.book(startTime,endTime)
# ["MyCalendarThree", "book", "book", "book", "book", "book", "book"]
# [[], [10, 20], [50, 60], [10, 40], [5, 15], [5, 10], [25, 55]]

In [44]:
calendar3 = MyCalendarThree()
print(calendar3.book(10, 20))
print(calendar3.book(60, 60))
print(calendar3.book(10, 40))
print(calendar3.book(5, 15))

1
1
2
3


<b> 370 range addition </b>

smart to apply the sweep line algo too. at the first glance, the quesiton looks simple, just do for loop for each item, but that is not efficient especially when N is big. the sweep line only cost O(N)

You are given an integer length and an array updates where updates[i] = [startIdxi, endIdxi, inci].

You have an array arr of length length with all zeros, and you have some operation to apply on arr. In the ith operation, you should increment all the elements arr[startIdxi], arr[startIdxi + 1], ..., arr[endIdxi] by inci.

Return arr after applying all the updates.

Input: length = 5, updates = [[1,3,2],[2,4,3],[0,2,-2]]  
Output: [-2,0,3,5,3]

In [47]:
from collections import defaultdict

def getModifiedArray(length: int, updates: List[List[int]]) -> List[int]:
    res = [0] * length
    temp_map = defaultdict(int)
    for s, e, val in updates:
        temp_map[s] += val
        temp_map[e+1] -= val
    cur = 0 # -> it is the vertical sweep line that sweep from starting point to the end point.
    for i in range(length):
        cur += temp_map[i]
        res[i] = cur 
    return res


In [46]:
getModifiedArray(5, [[1,3,2],[2,4,3],[0,2,-2]])

[-2, 0, 3, 5, 3]

<b> 1094 car pooling </b>

car only drive from west to east, and given the (num_of_passenger, from, to) and max capacity, return if the carpool works.

someone said it is similar to meeting room III, but meeting room does not allow overlap. here the sweep line algo actually is better than the merging interval way, because the merging interval way need to check when the passenger is off the car.

after update the dicitionary, even though no idea what is the max stop, we can just loop over the sorted key value.


In [48]:
from collections import defaultdict

def carPooling(trips: List[List[int]], capacity: int) -> bool:
    carpool = defaultdict(int)
    for num, start, to in trips:
        carpool[start] += num
        carpool[to] -= num
    ppl = 0
    for stop, num in sorted(carpool.items()):
        ppl += num
        if ppl > capacity:
            return False
    return True

In [49]:
carPooling([[2,1,5],[3,3,7]], 4)

False

<b> 452 Minimum Number of Arrows to Burst Balloons</b>  

an good one, when sorting the balloons actually sorting by its end, and shot at the end to ensure it can cover the max ballons. 

If sorting by the starting point, it actually not able to ensure all ballons within the overlapped the range will be covered

In [4]:
from typing import List
def findMinArrowShots(points: List[List[int]]) -> int:
    count = 0
    prev = None
    for start, end in sorted(points, key=lambda x:x[1]):
        if not prev or prev < start:
            # need to shoot the arrow and update the count
            count += 1
            # shoot as the furthest as it can so to increase the chance of bursting more ballons
            prev = end 
    return count

In [52]:
findMinArrowShots([[1,9], [4,5], [4,10], [7,14]])

2

In [5]:
findMinArrowShots([[1,6],[2,8],[7,12],[10,16]])

2

<b> 435 non-overlapping intervals </b>

similarly sorting by last value, so it can find MIN number of intervals that need to remove. And keep the one ends early will give latter one more room to not overlap
        

In [53]:

def eraseOverlapIntervals(intervals: List[List[int]]) -> int:
    intervals.sort(key=lambda x: x[1])
    prev = intervals[0][1]
    res = 0
    for start, end in intervals[1:]:
        if start < prev:
            res += 1
        else:
            prev = end
    return res


<b> 646 maximum length of pair chain </b>

same, sort by last item

In [55]:

def findLongestChain( pairs: List[List[int]]) -> int:
    # sort by last element, so it has more chance to get the longest chain
    pairs.sort(key=lambda x: x[1])
    prev = pairs[0][1]
    count = 1 # length of chain start by using first pair from the list
    for start, end in pairs[1:]:
        # no overlapping
        if prev < start:
            count += 1
            prev = end
    return count

In [56]:
findLongestChain([[-6,9],[1,6],[8,10],[-1,4],[-6,-2],[-9,8],[-5,3],[0,3]])

3

In [ ]:

def removeInterval(self, intervals: List[List[int]], toBeRemoved: List[int]) -> List[List[int]]:
    """
    sweep line way, actually better if toBeRemoved is a list of list, so multiple
    items need to be removed.
    """
    res = []
    tmp = []
    interval_map = defaultdict(int)
    for start, end in intervals:
        interval_map[start] += 1
        interval_map[end] -= 1
    interval_map[toBeRemoved[0]] -=1
    interval_map[toBeRemoved[1]] +=1
    acc_sum = 0
    for item, val in sorted(interval_map.items()):
        acc_sum += val
        if acc_sum == 1:
            tmp.append(item)
        elif tmp and acc_sum == 0:
            tmp.append(item)
            res.append(tmp)
            tmp = []
    return res



Another way of using stack and comparision, similar to the insert interval problems

In [61]:

def removeInterval(intervals: List[List[int]], toBeRemoved: List[int]) -> List[List[int]]:
    """
    using the way similar to insert interval problems
    """
    res = []
    for start, end in intervals:
        # the other scenarios is when  toBeRemoved [0] < start, end <= toBeRemoved[1], so completely removed
        if start < toBeRemoved[0]:
            # if no overlapping, [start, end] will be kept
            res.append([start, min(end, toBeRemoved[0])])
        # not elif, but if, so case like [[0, 5]] remove [2,3] can have correct output
        if end > toBeRemoved[1]:
            res.append([max(start, toBeRemoved[1]), end])
    return res

In [62]:
removeInterval([[0, 6]], [2, 3])

[[0, 2], [3, 6]]

or another easiler understanding way

In [64]:

def removeInterval(intervals: List[List[int]], toBeRemoved: List[int]) -> List[List[int]]:
    """
    using the way similar to insert interval problems
    """
    res = []
    for start, end in intervals:
        # no overlap
        if end <= toBeRemoved[0] or toBeRemoved[1] <= start:
            res.append([start, end])
        else:
            # some overlap
            if start < toBeRemoved[0]:
                res.append([start, toBeRemoved[0]])
            if toBeRemoved[1] < end:
                res.append([toBeRemoved[1], end])
            # the other cases are completely covered one and removed entirely
    return res


<b> 1229 meeting scheduler </b>  
given 2 list of time intervals, find the first one with common time interval with length at least x value.

way 1: using sweep line, the tricky part is how to locate the "end" time, cannot directly using value == 1 to locate the "end" point, but using additional prev_val and if value < prev_val meaning it is decreasing, as a way to find the "end" point.

way 2: sort the intervals first, then using two pointers to find the first overlapped interval. easy to understand, and it is quicker than the previous one.

time complexity of two ways are all O(NlogN) (sorting) while way1 is O(2Nlog(2N))


In [65]:

def minAvailableDuration(slots1: List[List[int]], slots2: List[List[int]], duration: int) -> List[int]:
    time_slots = defaultdict(int)
    for start, end in slots1:
        time_slots[start] += 1
        time_slots[end] -= 1
    for start, end in slots2:
        time_slots[start] += 1
        time_slots[end] -= 1

    common_dur = 0
    start = -1
    prev_count = 0
    for t, count in sorted(time_slots.items()):
        prev_count = common_dur
        common_dur += count
        if common_dur == 2:
            start = t
        elif common_dur < prev_count and start != -1: # -> here to locate the end time
            if t - start >= duration:
                return [start, start + duration]
            else:
                start = -1
    return []

In [66]:

def minAvailableDuration(slots1: List[List[int]], slots2: List[List[int]], duration: int) -> List[int]:
    slots1.sort()
    slots2.sort()
    m = len(slots1)
    n = len(slots2)
    i, j =0, 0
    while i < m and j < n:
        # find the first overlapping interval with len > duration
        # the first 2 if clause to skip the non-overlapped section can be removed, the slots1[i][1] < slots2[j][1]
        # comparision actually can cover this part
        if slots1[i][1] < slots2[j][0]:
            i += 1
        elif slots2[j][1] < slots1[i][0]:
            j += 1
        else:
            start = max(slots1[i][0], slots2[j][0])
            end = min(slots1[i][1], slots2[j][1])
            if end - start >= duration:
                return [start, start + duration]
            if slots1[i][1] < slots2[j][1]:
                i += 1
            else:
                j += 1
    return []

In [67]:
slots1= [[10,50],[60,120],[140,210]]
slots2 = [[0,15],[60,70]]
duration = 12
minAvailableDuration(slots1, slots2, duration)

[]

<b> 1288 remove covered intervals </b>

Given an array intervals where intervals[i] = [li, ri] represent the interval [li, ri), remove all intervals that are covered by another interval in the list.

The interval [a, b) is covered by the interval [c, d) if and only if c <= a and b <= d.

only remove fully coverred item. for example: [[1,4],[3,6],[2,8]] [3,6] is covered by [2, 8] so will be removed, result is 2

In [68]:

def removeCoveredIntervals(intervals: List[List[int]]) -> int:
    intervals.sort()
    prev_item = intervals[0]
    res = 0
    for item in intervals[1:]:
        # if prev_item fully cover item, item should be removed
        if  prev_item[0]<=item[0] and item[1] <= prev_item[1]:
            continue
        # item fully cover prev_item, prev_item should be removed
        elif item[0] <= prev_item[0] and prev_item[1] <= item[1]:
            prev_item = item
        else:
            res += 1
            prev_item = item
    return res+1

another way, sort the intervals by first item asc and second item desc order

In [69]:

def removeCoveredIntervals(intervals: List[List[int]]) -> int:
    # sort item by item[0] first, if item[0] is the same, sort by DESC order of item[0]
    # [[1,4], [1,2], [3, 8], [4,6]]
    intervals.sort(key=lambda x: (x[0], -x[1]))
    prev_item = intervals[0]
    res = 0
    for item in intervals[1:]:
        # not fully covered, no need to check item[0] because the sort already guaranteed item[0] is asc
        if item[1] > prev_item[1]:
            res+= 1
            prev_item = item
    return res+1

In [70]:
removeCoveredIntervals([[1,2], [1,4], [3, 8], [4,6]])

2

<b> 1353 maximum number of events that can. be attended </b>

although it is also meeting interval problems but quite different with others. it uses the pq to save the meeting ending time, so greedily attending meeting ending first to ensure more meetings can be attended.

Not able to simply sort by starting or ending time than take the meeting using the first available day that is not used, for example  
- [[1,2],[1,2],[3,3],[1,5],[1,5]] if sort by start time, and use first day available, it will be 1, 2, 3, 4 for 1,2],[1,2],[1,5], [1,5] and [3, 3] cannot be taken because day 3 is used.  -> but this is not true



In [ ]:
from heapq import heapify

def maxEvents(events: List[List[int]]) -> int:
    # sort events by starting time
    events.sort()
    total_days = max(event[1] for event in events)
    pq = [] # save the ending time of event
    event_ind = 0
    res = 0
    n = len(events)
    day = 1
    while day <= total_days:
        # if no events for today, jump to next starting day
        if event_ind < n and not pq:
            day = events[event_ind][0]

        # if event starting day < today, meaning possible to attend this event today
        # add the possible events to pq
        # only means possible because the event end date can < day too
        while event_ind < n and events[event_ind][0] <= day:
            heapq.heappush(pq, events[event_ind][1])
            event_ind += 1

        # remove event that cannot be attended because the ending day is < current day
        while pq and pq[0] < day:
            heapq.heappop(pq)

        # now the events left in pq are attendable
        # in order of whichever end first
        # for a single day, only can attend one meeting, so not a while loop
        if pq:
            heapq.heappop(pq)
            res += 1
        day += 1
    return res


<b> 1589  Maximum Sum Obtained of Any Permutation</b>

a goold sweep line algo implementation.

first get the frequency of each index, then order by the freq, the ind with largest freq should pair with the largest value from the nums array, in order to get the max sum

In [3]:
from typing import List
def maxSumRangeQuery(nums: List[int], requests: List[List[int]]) -> int:
    """sweep line"""
    # no need to have a defaultdict but using an array with len len(nums)+1
    # because the requests value are index of nums, so the largest val would be len(nums)+1
    n = len(nums)
    # +1 here because we want to -1 value of freq for next item of end index
    req_count = [0] * (n+1) 
    for start, end in requests:
        req_count[start] += 1
        req_count[end+1] -= 1 # end+1 so need to create an array with len n+1

    for i in range(1, n+1):
        req_count[i] += req_count[i-1]

    # req_count stores the frequency of requests for each index
    # the index with highest freq should pair with the largest value from nums array
    req_count.sort(reverse = True)
    nums.sort(reverse = True)
    res = 0
    # n+1 element at req_count should always be 0, no need
    for i in range(n):
        res += nums[i] * req_count[i]
    # print("req_count", req_count)
    # print("nums", nums)
    # print("res", res)
    return res % (10**9 + 7)



In [4]:
nums = [1,2,3,4,5,10]
requests= [[0,2],[1,3],[1,1]]
maxSumRangeQuery(nums, requests)

47

index 1 has the highest frequency, so the largest value from nums array should be at index 1. the same as the largest freq value should pair with largest num value,so sort the req_count and nums accordinly, but no need to care about what is exact index mapping

<b> 1943 describe the painting </b>

sweep line too.  
using defaultdict to add and minus the color at start and end respectively (because the end is not included)  
then loop over each index with color value accumulated change, each index value is actually either a start or an end of a painting segment.

edge case is when mixed painting color is 0, that means no painting happen at that index range, no need to include it

it is easier to visualize it by drawing the line in the index range with color code



In [7]:
from collections import defaultdict
def splitPainting(segments: List[List[int]]) -> List[List[int]]:
    color_map = defaultdict(int)
    for start, end, color in segments:
        color_map[start] += color
        color_map[end] -= color
    res = []
    sorted_color_map = sorted(color_map.items())
    # print("sorted_color_map", sorted_color_map)
    prev_ind, prev_color = sorted_color_map[0]
    for ind, color in sorted_color_map[1:]:
        if prev_color: # no painting happen, no need to take into consideration
            res.append([prev_ind, ind, prev_color])
        prev_ind = ind
        prev_color += color
    return res



In [8]:
splitPainting([[1,7,9],[6,8,15],[8,10,7]])

[[1, 6, 9], [6, 7, 24], [7, 8, 15], [8, 10, 7]]

<b> 2251 number of flowers in full bloom </b>

sweep line + binary search solution, using the graph from the problems can help make the problems more intuitive

In [9]:
from collections import defaultdict
from bisect import bisect_left

def fullBloomFlowers(flowers: List[List[int]], people: List[int]) -> List[int]:
    "sweep line, with bisect"
    flower_schedule = defaultdict(int)
    for start, end in flowers:
        flower_schedule[start] += 1
        flower_schedule[end+1] -= 1
    prev = None
    schedule = sorted(flower_schedule.keys())

    for t in schedule:
        if prev:
            flower_schedule[t] += flower_schedule[prev]
        prev = t
    res = []
    for p in people:
        # find the index of p in the keys of flower_schedule
        # using binary search to find the key, because the date (key) kept in flower_schedule only stores
        # the start, end date, not a full schedule, so it is descrete list
        # the people arrive should find the nearest starting schedule of the blossom
        if p in flower_schedule:
            res.append(flower_schedule[p])
        else:
            ind = bisect_left(schedule, p)
            res.append(flower_schedule[schedule[ind-1]])
    return res



In [10]:
fullBloomFlowers([[1,6],[3,7],[9,12],[4,13]], [2,3,7,11])

[1, 2, 2, 2]

the intermediate value is   
schedule [1, 3, 4, 7, 8, 9, 13, 14]  
defaultdict(<class 'int'>, {1: 1, 7: 2, 3: 2, 8: 1, 9: 2, 13: 1, 4: 3, 14: 0})   

the above solution can be further improved. by not actually care about the the key in the map, but using the index

imagine the flower chart in the problem desc.  
when a person comes at day 2, because we have kept record of start, end day only, so the schedule we have only has day 1, or day 3, if people arrive at day 2, it should look at the flower counted at day 1, because day 3 is not yet there.

so we can leverage the bisect function to find the index where day 2 should exist, and -1 is the actual valud day

[1,3, 4] for day 3, it should get the flowers saved for day 3 since 3 is availabe in schedule, bisect returns index 2, index 1 is actually where day 3 is in the schedule list   
[1,3,4] for day 2, it should get the flowers saved for day 1 since next day 3 is not here yet for day 2. bisect returns index 1, but 1-1=0, mapping to day 1 is what ppl arrive at day 2 should look at

In [11]:
from collections import defaultdict
from bisect import bisect_left, bisect

def fullBloomFlowers(flowers: List[List[int]], people: List[int]) -> List[int]:
    "sweep line, with bisect"
    flower_schedule = defaultdict(int)
    for start, end in flowers:
        flower_schedule[start] += 1
        flower_schedule[end+1] -= 1
    schedule = sorted(flower_schedule.keys())
    flower_count = [flower_schedule[i] for i in schedule]
    # accumulate the flower_count
    for i in range(1, len(flower_count)):
        flower_count[i] += flower_count[i-1]
    print("flower_count", flower_count)

    res = []
    for p in people:
        # the schedule list and flower_count list are in order
        # so the index-1 retrieved using bisect happens to be the location
        # that ppl appear should check the flower count
        # using a graph is more intutive
        ind = bisect(schedule, p)
        res.append(flower_count[ind-1])

    return res



<b> 1326 mininum number of taps to open to water a garden </b>

smart to use a list to keep track of the furthest position that index i can reach, which means the ith index indicates i is the starting point, the value at ith point: list[i] is the furthest position that can be reached for this tap.

then using the sweep line algo to loop over i to n+1, also have current tap's furthest position and next tap's furthest position. comparing current tap's furthest position with i to determine if should open next tap

In [12]:

def minTaps(n: int, ranges: List[int]) -> int:
    # a list to store the furthest position 
    # index of the list water_range is the starting point
    # value at this index is the ending point
    # it could be that multiple tap's starting point is the same but ending point is different
    # in the list we keep the largest ending point
    water_range = [0] * (n+1)
    for i in range(n+1):
        left = max(0, i - ranges[i])
        right = min(n, i + ranges[i])
        water_range[left] = max(water_range[left], right)
    # print("water_range", water_range)
    # sweep line, loop from 0 to n
    curr_range = water_range[0] # furthest point if starting point is 0
    next_range = 0 # which tap should be open next
    res = 0
    for i in range(n+1):
        if i > curr_range:
            # position i cannot be covered
            return -1<b
        elif i == curr_range:
            # current tap's range is reaching the end, need to find the next one
            next_range = max(next_range, water_range[i])
            res += 1
            curr_range = next_range
            # reset next_range
            next_range = 0
        else:
            # current tap's range is still good, i is still covered by current tap's range
            # but good to keep check of the next tap 
            # ensure to find next good one has the largest coverage
            next_range = max(next_range, water_range[i])
    return res






In [13]:
minTaps(5, [3,4,1,1,0,0])

1

<b> 45 Jump Game II </b>

someone said it is similar to the problem of 1326, the solution do looks a bit similar 


In [14]:
def jump(nums: List[int]) -> int:
    cur_furthest_jump = 0
    next_jump = 0
    res = 0
    n = len(nums)
    # i to n-2, not including the last index, because if i at n-1, it means already reach the end, no need
    # to check where last ind n-1 can take further
    for i in range(n-1):
        if i > cur_furthest_jump:
            return -1
        # keep track of the furthest jump that can be reached at each step
        next_jump = max(next_jump, i + nums[i])
        if i == cur_furthest_jump:
            # change another jump
            # but does not mean jump at i for nums[i]
            # because the next_jump value is updated every time
            # it can [2,3,1,1,4], num[0] = 2, at ind 2, the nums[0] is used up, 
            # next_jump value is actually 1+3 because next_jump keep updated to have the largest value
            # so even though at ind 2 the furtherst jump value now uses nums[1] instead of nums[2]
            res += 1
            cur_furthest_jump = next_jump
            # no need to reset next_jump
    return res

In [15]:
jump([2,3,1,1,4])

2

<b> 1306 Jump game III </b>

related problem with 45, but not using sweep line, using recursion instead. A smart way to flip the visited value as negative, and check to allow recursion move forward only when the value is not negative which means not visited before, to avoid deadloop

In [16]:

def canReach(arr: List[int], start: int) -> bool:
    """recursion"""
    # start is a valid index, not out of bound
    # position start is not visited before
    if 0 <= start < len(arr) and arr[start] >= 0:
        # mark visited position
        arr[start] = - arr[start]
        return arr[start] == 0 or canReach(arr, start + arr[start]) or canReach(arr, start - arr[start])

    # base case
    return False

In [17]:
canReach([4,2,3,0,3,1,2], 5)

True

<b> 1871 jump game VI </b>

DP + sliding window, the solution is not too intuitive at the beginning

In [19]:

def canReach( s: str, minJump: int, maxJump: int) -> bool:
    # dp[i] means if ith position is reachable from start
    # end result is dp[-1]
    dp = [c == "0" for c in s]
    # for dp[i], any one between dp[i-maxJump: i-minJump] is true, then dp[i] should be true
    # i-maxJump is the left most position where after jumping maxJump step can reach position i
    # i-minJump is the right most position where after jumping minJump step that can reach the position i

    count = 0 # count the True element within the jump window
    for i in range(1, len(s)):
        # sliding window, every step only need to add new position, and left out old position not within the [minJump, maxjump] window
        if i - minJump >=0 :
            # i-minJump pos is the closet position that jumping minJump step and reach i
            # if min_jump = 3, ind 0 -> jump 3 -> reach ind 3, so ind 1 and 2 are never counted
            count += dp[i-minJump]
        if i- maxJump -1>=0:
            # i-maxJump is the left most position that jumping step of maxJump can reach i
            # i-maxJump should still be included, but one position left is the one should be removed
            # that is why i - maxJump - 1
            count -= dp[i-maxJump - 1]

        # True is 1, False is 0, so if any of position in the jumping window has True the count will > 0
        # original dp[i] already check if s[i] == '0'
        # i is reachable only when ith is 0 and some previous step can reach this position
        dp[i] = dp[i] and count > 0

    return dp[-1]      

In [20]:
canReach("01101110", 2 ,3)

False

<b>1851 Minimum Interval to Include Each Query </b>
- sort the queries, and the intervals, to make it easier to get the interval by order, using [::-1] and pop()
- loop over each query, and populate the heap queue with element [size of interval, ending position] for later usage
- need to clean up queue because after each query comparision, it can happen that the left intervals do not satisfy the constraint to contain the query
- the first item from the heap queue is the interval that satisfy the contraint with smallest interval size and contain the query

Time complexity:
- n: intervals size, q: query size
- sort: O(nlogn + qlogq)
- heap queue operaitons: O(logN)
- total sort operations complexity takes the most time, so O(NlogN + QlogQ)

space complexity: O(n+q)

In [ ]:
import heapq

def minInterval(intervals: List[List[int]], queries: List[int]) -> List[int]:
    # sort intervals by first element, then second element, 
    # because we want to use the interval from left to right but pop() function get the 
    # element from right to left, hence [::-1]
    sorted_intervals = sorted(intervals)[::-1]
    queue = [] # store [interval_size, end]
    res = {}
    for q in sorted(queries):
        while sorted_intervals and sorted_intervals[-1][0] <= q:
            start, end = sorted_intervals.pop()
            if q <= end:
                # find the interval contains with q
                heapq.heappush(queue, [end-start+1, end])
        # because of using the item of queue for query in later step, it could be that
        # the first item in queue is no longer valid for the next query, so need a cleanup
        while queue and queue[0][1] < q:
            heapq.heappop(queue)

        # not the first item in queue is the one with shortest interval size but contains q
        # here directly reading first item from queue instead of pop it out so it can be used
        # by other query, which means the interval can be used by multiple query
        if queue:
            res[q]=queue[0][0]
        else:
            res[q] = -1
    # need to return the result in the same order of queries
    return [res[q] for q in queries]

<b> 1024 video stitching </b>

similar to interval, but try to find the number of least intervals and after combine them it can contain the range from 0 to target value
- still using 2 variables, one to save the prev end, another to save the current reachable end
- do the comparison, and only add res by 1 when a merge is first initialized
- the max() function is used to update / extend the current reachable end,

In [26]:

def videoStitching(clips: List[List[int]], time: int) -> int:

    prev_end = -1
    curr_end = 0 # to find range[0, time], starting point
    res = 0
    for i, j in sorted(clips):
        # if already meet the end, or there is gap that next starting point cannot connect with current end
        # stop the iteration, latter case is false case
        if curr_end >= time or i > curr_end:
            break
        # as long as the starting i is within this range, we can add the count
        # because the curr_end = max(curr_end, j) will update the new reachable end
        # even if the current [i,j] is not the ideal one to be included
        # but because i indicates that the next one could be the one to extend the range
        # for exampoe [0, 2],[1,7],[1,9] -> the count increase when it meets[1,7] eventhough
        # [1,7] is not the one we should select but [1,9], but it is okay
        # in next if check, the i is 1, but the prev_end is 2, res wont increase
        # however. the curr_end will be updated to 9
        if prev_end < i <= curr_end: # -> can merge, so add 1 count
            res += 1
            prev_end = curr_end
        curr_end = max(curr_end, j) # -> update the new end, it is valid because the nonvalid case is rule out in the break clause, and the interval is sorted

    return res if curr_end >= time else -1

In [25]:
videoStitching([[0, 2],[1,7],[1,9]], 9)

2

<b> 3397 Maximum number of distinct elements after operations </b>

[n-k, n+k] is the range or interval, using a variable: last_val_used to track which interval it is in right now.it uses the brute force way to always select the smallest possible value

In [ ]:
import math
def maxDistinctElements(nums: List[int], k: int) -> int:
    """
    if finding the max count of common elements, this problem can be solved by sweep line method

    time complexity: O(NlogN) because of sorting
    """
    nums.sort()
    res = 0
    last_val_used = -math.inf
    for n in nums:
        left = n - k
        right = n + k
        if last_val_used < left:
            # choose left as the val
            last_val_used = left
            res += 1
        elif last_val_used < right:
            # last value used is within the range of [left, right]
            last_val_used += 1
            res += 1
    return res



<b> 3394 check if grid can be cut into sections </b>

a transformation question of interval merge.  
check the x axis and y axis separately, see if the count of non-overlapping intervals >=3. either one satisfy the requirements can return True

##### REMEMBER
when updating the range, using merged_list[-1] = <b>max</b>(merged_list[-1], end)

In [4]:

def checkValidCuts(n: int, rectangles: List[List[int]]) -> bool:
    """
    first check from x axis, sort rectangles by x, merge them if overlapped, if the length of result >=3, it is possible to cut vertically
    otherwise check from the y axis, sort by y, merge if overlapped and check the result list
    """
    x_list = sorted([[r[0],r[2]] for r in rectangles])
    y_list = sorted([[r[1],r[3]] for r in rectangles])

    # check x_list to see if horizontal cut is valid
    def helper(ind_list, merged_list):
        for start, end in ind_list:
            if not merged_list or merged_list[-1]<=start:
                merged_list.append(end)
            else:
                merged_list[-1] = max(merged_list[-1], end)
        if len(merged_list)>=3:
            return True
        return False
    merged_list = []
    is_vertical_cut = helper(x_list, merged_list)
    if is_vertical_cut: return True
    
    merged_list = []
    is_horizontal_cut = helper(y_list, merged_list)

    return is_horizontal_cut
        
    

In [5]:
checkValidCuts(4, [[0,0,1,4],[1,0,2,3],[2,0,3,3],[3,0,4,3],[1,3,4,4]])

False

<b> get the interval with max freq </b>

given a list of interval, return the start and end time of the most frequent intervals

for example:

[[2,5], [3,6]] -> [3, 5]  
[[2,5], [3,6], [3,7], [5, 8]] -> [5, 5]

In [13]:
from collections import defaultdict
def get_max_freq_interval(intervals: List[List[int]]):
    freq_map = defaultdict(int) # {time: count}
    for start, end in intervals:
        freq_map[start] += 1
        freq_map[end+1] -= 1
    freq = 0 # accumulated freq count
    max_freq = 0
    start_max = 0
    end_max = 0
    acc_freq = {}
    for t in sorted(freq_map):
        freq += freq_map[t]
        acc_freq[t] = freq
        if freq > max_freq:
            start_max = t
            max_freq = freq
        elif freq < max_freq and end_max<start_max:
            end_max = t-1
    print(acc_freq)
    return (start_max, end_max)

In [14]:
get_max_freq_interval([[2,5], [3,6], [3,7], [5, 8]])

{2: 1, 3: 3, 5: 4, 6: 3, 7: 2, 8: 1, 9: 0}


(5, 5)

In [16]:
get_max_freq_interval([[2,7], [3,7], [3,7], [5, 8]])

{2: 1, 3: 3, 5: 4, 8: 1, 9: 0}


(5, 7)